# ARC-v0.24 — Post-Primary Family-Balanced Robustness Audit

**Purpose.** Re-analyze the completed ARC-v0.24 MS MARCO endpoint table with equal weight assigned to the two feedback-policy families (`mean` and `softmax`).

This is a **post-primary robustness audit only**. It does **not** rerun retrieval, encoding, FAISS, or trajectories, and it does not redefine the frozen primary hypotheses.

The audit requires either:

- an in-memory completed `endpoints` DataFrame, or
- `v024_validation-full_endpoints.parquet` from the completed 3,490-query validation run.

It computes query-level 10,000-replicate bootstrap confidence intervals for:

1. mean-family H3abs,
2. softmax-family H3abs,
3. equal-family-weighted H3abs = 0.5 × mean + 0.5 × softmax,
4. paired representation-minus-nprobe equal-family-weighted contrast.


In [ ]:
# ARC-v0.24 post-primary family-balanced robustness audit
# NO retrieval / NO encoding / NO FAISS / NO GPU required.
# Requires the completed v024_validation-full_endpoints.parquet, or an in-memory
# DataFrame named `endpoints` with the same columns.

from pathlib import Path
import json
import numpy as np
import pandas as pd

SEED = 20260824
BOOTSTRAP_REPS = 10_000
MEASURE = "H3_abs_slope"

REQUIRED = {
    "query_id", "mechanism", "method", "config_key", MEASURE
}


def _load_endpoints():
    # 1) Reuse in-memory completed endpoint table if present.
    g = globals()
    if "endpoints" in g and isinstance(g["endpoints"], pd.DataFrame):
        df = g["endpoints"].copy()
        if REQUIRED.issubset(df.columns):
            print("Using in-memory `endpoints` DataFrame.")
            return df, None

    # 2) Reuse explicit ENDPOINT_PATH if defined.
    if "ENDPOINT_PATH" in g:
        p = Path(g["ENDPOINT_PATH"])
        if p.is_file():
            print("Loading:", p)
            return pd.read_parquet(p), p

    # 3) Search only for the compact completed endpoint artifact.
    roots = [
        Path("/content/drive/MyDrive"),
        Path("/content"),
        Path.cwd(),
    ]
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        try:
            candidates.extend(root.rglob("v024_validation-full_endpoints.parquet"))
        except Exception:
            pass

    # Deduplicate + prefer Drive paths.
    uniq = []
    seen = set()
    for p in candidates:
        s = str(p.resolve())
        if s not in seen:
            seen.add(s)
            uniq.append(p)

    if not uniq:
        raise FileNotFoundError(
            "No completed v024_validation-full_endpoints.parquet found and no "
            "in-memory `endpoints` DataFrame is available. This audit does NOT "
            "need retrieval reruns; provide/restore only the completed endpoint parquet."
        )

    uniq.sort(key=lambda p: ("/content/drive/" not in str(p), -p.stat().st_mtime))
    p = uniq[0]
    print("Loading:", p)
    return pd.read_parquet(p), p


def _query_family_table(df):
    # First average all policies within query x mechanism x feedback family.
    qfam = (
        df.groupby(["query_id", "mechanism", "method"], as_index=False)[MEASURE]
          .mean()
    )

    counts = (
        df[["mechanism", "method", "config_key"]]
        .drop_duplicates()
        .groupby(["mechanism", "method"])
        .size()
        .rename("n_policies")
        .reset_index()
    )

    print("\nFrozen policy counts:")
    print(counts.to_string(index=False))

    # Need both mean and softmax for every query/mechanism.
    wide = qfam.pivot_table(
        index=["query_id", "mechanism"],
        columns="method",
        values=MEASURE,
        aggfunc="mean",
    ).reset_index()

    missing = [c for c in ["mean", "softmax"] if c not in wide.columns]
    if missing:
        raise RuntimeError(f"Missing feedback families in endpoints: {missing}")

    wide["family_balanced_H3abs"] = 0.5 * (
        wide["mean"].astype(float) + wide["softmax"].astype(float)
    )
    return qfam, wide, counts


def _bootstrap_mean(x, rng, reps=BOOTSTRAP_REPS):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    n = len(x)
    if n == 0:
        raise RuntimeError("No finite observations.")
    point = float(x.mean())
    boot = np.empty(reps, dtype=np.float64)
    for b in range(reps):
        boot[b] = x[rng.integers(0, n, size=n)].mean()
    lo, hi = np.quantile(boot, [0.025, 0.975])
    return point, float(lo), float(hi), n


def main():
    df, source_path = _load_endpoints()
    missing = REQUIRED - set(df.columns)
    if missing:
        raise RuntimeError(f"Endpoint table missing columns: {sorted(missing)}")

    # Hard guards against accidentally analyzing smoke/partial data.
    nq = df["query_id"].astype(str).nunique()
    print("Endpoint rows:", f"{len(df):,}")
    print("Unique queries:", nq)
    if nq != 3490:
        raise RuntimeError(
            f"Expected the completed 3,490-query validation run, found {nq}. "
            "Do not use a partial/smoke table for the manuscript audit."
        )

    qfam, wide, counts = _query_family_table(df)
    rng = np.random.default_rng(SEED + 2411)

    rows = []
    for mech in sorted(wide["mechanism"].unique()):
        sub = wide[wide["mechanism"].eq(mech)]
        for estimand in ["mean", "softmax", "family_balanced_H3abs"]:
            point, lo, hi, n = _bootstrap_mean(sub[estimand], rng)
            rows.append({
                "mechanism": mech,
                "estimand": estimand,
                "mean": point,
                "ci95_low": lo,
                "ci95_high": hi,
                "n_queries": n,
            })

    summary = pd.DataFrame(rows)

    # Paired representation-minus-nprobe family-balanced contrast.
    pair = wide.pivot_table(
        index="query_id",
        columns="mechanism",
        values="family_balanced_H3abs",
        aggfunc="mean",
    ).dropna()
    if not {"representation", "nprobe"}.issubset(pair.columns):
        raise RuntimeError("Could not form paired representation/nprobe query table.")
    delta = pair["representation"] - pair["nprobe"]
    point, lo, hi, n = _bootstrap_mean(delta, rng)
    paired_row = {
        "mechanism": "representation-minus-nprobe",
        "estimand": "family_balanced_H3abs_paired_delta",
        "mean": point,
        "ci95_low": lo,
        "ci95_high": hi,
        "n_queries": n,
    }
    summary = pd.concat([summary, pd.DataFrame([paired_row])], ignore_index=True)

    print("\n================ FAMILY-BALANCED QUERY BOOTSTRAP ================")
    print(summary.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

    rep = summary[(summary.mechanism == "representation") &
                  (summary.estimand == "family_balanced_H3abs")].iloc[0]
    npb = summary[(summary.mechanism == "nprobe") &
                  (summary.estimand == "family_balanced_H3abs")].iloc[0]
    dlt = summary[summary.estimand == "family_balanced_H3abs_paired_delta"].iloc[0]

    gate = {
        "status": "ARC_V024_POST_PRIMARY_FAMILY_BALANCED_AUDIT",
        "n_queries": 3490,
        "bootstrap_reps": BOOTSTRAP_REPS,
        "estimand": "equal 0.5 weight to mean-family query mean and softmax-family query mean",
        "representation": {
            "mean": float(rep["mean"]),
            "ci95": [float(rep["ci95_low"]), float(rep["ci95_high"])],
            "positive": bool(rep["mean"] > 0),
            "ci_excludes_zero_positive": bool(rep["ci95_low"] > 0),
        },
        "nprobe": {
            "mean": float(npb["mean"]),
            "ci95": [float(npb["ci95_low"]), float(npb["ci95_high"])],
            "negative": bool(npb["mean"] < 0),
            "ci_excludes_zero_negative": bool(npb["ci95_high"] < 0),
        },
        "paired_representation_minus_nprobe": {
            "mean": float(dlt["mean"]),
            "ci95": [float(dlt["ci95_low"]), float(dlt["ci95_high"])],
            "positive": bool(dlt["mean"] > 0),
            "ci_excludes_zero_positive": bool(dlt["ci95_low"] > 0),
        },
        "post_primary_robustness_only": True,
        "does_not_redefine_frozen_primary_hypotheses": True,
    }

    print("\n================ GATE ================")
    print(json.dumps(gate, indent=2))

    # Save next to endpoint artifact when possible; otherwise into current OUT/cwd.
    if source_path is not None:
        save_dir = source_path.parent
    elif "OUT" in globals() and Path(globals()["OUT"]).is_dir():
        save_dir = Path(globals()["OUT"])
    else:
        save_dir = Path.cwd()

    csv_path = save_dir / "v024_postprimary_family_balanced_h3abs.csv"
    json_path = save_dir / "v024_postprimary_family_balanced_gate.json"
    summary.to_csv(csv_path, index=False)
    json_path.write_text(json.dumps(gate, indent=2))

    print("\nSaved:")
    print(csv_path)
    print(json_path)

    return summary, gate


if __name__ == "__main__":
    FAMILY_BALANCED_SUMMARY, FAMILY_BALANCED_GATE = main()


## Expected outputs

The notebook writes two compact artifacts next to the endpoint parquet when possible:

- `v024_postprimary_family_balanced_h3abs.csv`
- `v024_postprimary_family_balanced_gate.json`

For manuscript hardening, the most important quantities are the `family_balanced_H3abs` confidence interval for `representation` and the paired `representation-minus-nprobe` confidence interval.
